# Level 6 — Thử nghiệm xếp hạng chú thích bằng quan hệ RelTR

**Nhóm 02 — CNTT-KLCN219 | Trạng thái: prototype, chưa có kết quả COCO.**

Notebook này thay cho hai file Python demo cũ. Upload **chỉ file `.ipynb` này** lên Colab;
chọn **CPU** và chạy lần lượt hoặc Run all sau khi kiểm tra cấu hình ở mục 2.

Notebook tự chứa mã đọc CSV RelTR thật, mở rộng vị ngữ, kiểm tra chiều S–P–O theo quy tắc từ vựng,
và xếp hạng ứng viên **riêng cho từng ảnh**. Không cần upload hai file `.py` cũ.

**Phạm vi:** mặc định kiểm tra dữ liệu Karpathy Val. Nếu cung cấp caption ứng viên và điểm baseline
từ Level 5A, notebook xuất điểm xếp hạng mới trước MMR. Không tạo FAISS, không thay kho tri thức,
không train decoder, không sinh caption và không chấm Test. Các module thực thi Level 5A chưa có
trong bộ file này nên không tự động chạy train → retune → test.

Tài liệu Word cũ đi kèm là tài liệu lịch sử có nhận định cần hiệu chỉnh; dùng hướng dẫn trong notebook
cho mã hiện tại. Mọi mốc Level 1–4 được giữ nguyên, không phát sinh số liệu cải thiện giả định.

## 1. Chuẩn bị Colab
Cài NLTK và tải WordNet. Khi yêu cầu WordNet mà thiếu dữ liệu, notebook dừng với lỗi rõ ràng;
không âm thầm coi từ điển viết sẵn là WordNet. Không cần GPU cho phần này.

In [ ]:
import sys, subprocess
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass
if not IN_COLAB:
    raise RuntimeError('Cell thiết lập dành cho Google Colab. Upload notebook lên Colab rồi chạy lại.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nltk==3.9.1'])
import nltk
if nltk.__version__ != '3.9.1':
    raise RuntimeError('NLTK cũ còn trong RAM. Runtime → Restart session rồi chạy lại.')
for resource in ['wordnet', 'omw-1.4']:
    if not nltk.download(resource, quiet=True):
        raise RuntimeError(f'Không tải được {resource}; kiểm tra mạng và chạy lại cell này.')
from google.colab import drive
drive.mount('/content/drive')
print('Môi trường sẵn sàng. NLTK:', nltk.__version__)

## 2. Cấu hình đường dẫn và thí nghiệm
`DATA_ROOT` mặc định giữ đúng hai cấp `Data/Data`. Sửa nếu Drive của bạn có tên khác.
`RELTR_CSV` phải là file Val thật, schema `image,prediction,score`; nếu đang có ZIP thì giải nén trước.

- `CANDIDATES_JSONL = None`: chỉ kiểm tra dữ liệu RelTR, lưu thống kê và ví dụ.
- Có file ứng viên: điền đường dẫn JSONL theo mục 7 để chạy reranking.
- `ALPHA=0.20` là giá trị thử ban đầu, **chưa được tối ưu hay chứng minh tốt hơn**.
- Dùng `ALPHA=0` để kiểm tra nhánh mới không thay đổi điểm và thứ hạng baseline.
- Giữ MMR, cổng fusion và 5 knowledge tokens ở pipeline Level 5A khi tích hợp.

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/DataKhoaLuan/Data')
SPLIT = 'val'  # Notebook thử nghiệm chỉ chạy Val; không dùng Test để chọn phương pháp.
RELTR_CSV = DATA_ROOT / 'reltr_inputs' / 'coco_val_reltr_detection.csv'
CANDIDATES_JSONL = None  # Ví dụ: DATA_ROOT / 'baseline_level6/inputs/candidates_val.jsonl'
ALPHA = 0.20
MIN_CONFIDENCE = 0.50
USE_WORDNET = True
EXPECTED_VAL_IMAGES = 5000
OUTPUT_ROOT = DATA_ROOT / 'baseline_level6' / 'Tan_Dung_RelTR_3Quanhe' / 'val'
assert SPLIT == 'val', 'Chốt phương pháp trên Val trước khi xây quy trình Test chính thức.'
assert 0 <= ALPHA <= 1 and 0 <= MIN_CONFIDENCE <= 1
print('CSV:', RELTR_CSV)
print('Chế độ:', 'inspect' if CANDIDATES_JSONL is None else 'rerank')

## 3. Mã xử lý quan hệ
Cell dưới chứa toàn bộ module mới để có thể kiểm tra và chạy độc lập.

Điểm quan hệ là tỷ lệ bộ ba khớp, có trọng số confidence của RelTR. Bộ so khớp yêu cầu
chủ thể → quan hệ → đối tượng theo đúng thứ tự trong cùng mệnh đề; giữ các từ `on`, `in`, `under`.
Mở rộng vị ngữ dùng các lemma cùng synset động từ đầu tiên, không dùng troponym như từ đồng nghĩa.

**Giới hạn:** đây là heuristic từ vựng, không phải bộ phân tích cú pháp hoặc scene graph encoder.
Câu bị động, đại từ, phối hợp phức tạp và từ đa nghĩa có thể bị bỏ sót hoặc khớp sai.
WordNet sense đầu tiên và các nhãn vật thể quy về `person` có thể gây nhiễu. Cần kiểm tra định tính
trên Val; không tuyên bố đã hiểu đầy đủ mọi quan hệ S–P–O.

In [ ]:
"""Lv6 prototype: ordered lexical relation matching, not a scene-graph parser."""
import csv
import hashlib
import json
import math
import re
from collections import defaultdict
from pathlib import Path

_WN = None
_LEMMATIZER = None
_ALIASES = {
    'man': 'person', 'woman': 'person', 'boy': 'person', 'girl': 'person',
    'child': 'person', 'people': 'person', 'men': 'person', 'women': 'person',
    'children': 'person', 'bike': 'bicycle', 'bikes': 'bicycle',
    'bicycles': 'bicycle', 'dogs': 'dog', 'cats': 'cat', 'horses': 'horse',
    'riding': 'ride', 'rides': 'ride', 'ridden': 'ride', 'rode': 'ride',
    'chasing': 'chase', 'chases': 'chase', 'chased': 'chase',
    'holding': 'hold', 'holds': 'hold', 'held': 'hold',
    'wearing': 'wear', 'wears': 'wear', 'wore': 'wear',
    'sitting': 'sit', 'sits': 'sit', 'sat': 'sit',
    'standing': 'stand', 'stands': 'stand', 'stood': 'stand',
    'playing': 'play', 'plays': 'play', 'played': 'play',
    'eating': 'eat', 'eats': 'eat', 'ate': 'eat',
    'carrying': 'carry', 'carries': 'carry', 'carried': 'carry',
}
_PHRASE_ALIASES = {
    'next to': ('beside', 'alongside'),
    'above': ('over',), 'below': ('under', 'beneath'),
}


def configure_wordnet(enabled=True):
    """Fail explicitly when requested WordNet data are unavailable."""
    global _WN, _LEMMATIZER
    if not enabled:
        _WN, _LEMMATIZER = None, None
        return
    from nltk.corpus import wordnet
    from nltk.stem import WordNetLemmatizer
    wordnet.ensure_loaded()
    _WN, _LEMMATIZER = wordnet, WordNetLemmatizer()


def canonical_image_id(value):
    text = str(value).strip().replace('\\', '/').rsplit('/', 1)[-1]
    if re.fullmatch(r'\d+', text):
        return str(int(text))
    match = re.fullmatch(r'COCO_(?:train|val|test)2014_(\d+)\.jpg', text, re.I)
    if match:
        return str(int(match.group(1)))
    raise ValueError(f'image_id/filename COCO không hợp lệ: {value!r}')


def words(text):
    result = []
    for token in re.findall(r"[a-z]+(?:'[a-z]+)?", str(text).lower().replace('_', ' ')):
        token = _ALIASES.get(token, token)
        if _LEMMATIZER is not None:
            token = _LEMMATIZER.lemmatize(token, pos='v')
            token = _LEMMATIZER.lemmatize(token, pos='n')
        result.append(_ALIASES.get(token, token))
    return tuple(result)


def load_reltr_triplets(csv_path, min_conf=0.5):
    """Accept real image/prediction/score CSV and split-column CSV; retain empty images."""
    if not 0 <= min_conf <= 1:
        raise ValueError('min_conf phải nằm trong [0,1]')
    table, seen = {}, defaultdict(set)
    stats = {'rows': 0, 'no_relation': 0, 'below_threshold': 0, 'duplicates': 0}
    with Path(csv_path).open(encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        names = {name.strip().lower(): name for name in (reader.fieldnames or [])}
        def pick(*options):
            return next((names[name] for name in options if name in names), None)
        image_col = pick('image_id', 'image', 'file_name')
        pred_col = pick('prediction')
        sub_col, rel_col, obj_col = pick('subject', 's'), pick('predicate', 'relation', 'rel', 'p'), pick('object', 'o')
        score_col = pick('score', 'confidence', 'conf', 'reltr_score')
        if not image_col or not score_col or not (pred_col or (sub_col and rel_col and obj_col)):
            raise ValueError(f'CSV thiếu schema RelTR hỗ trợ: {reader.fieldnames}')
        for row_number, row in enumerate(reader, 2):
            try:
                image_id = canonical_image_id(row[image_col])
                table.setdefault(image_id, [])
                stats['rows'] += 1
                score = float(row[score_col])
                if not math.isfinite(score) or not 0 <= score <= 1:
                    raise ValueError('confidence phải hữu hạn và thuộc [0,1]')
                if pred_col:
                    prediction = row[pred_col].strip()
                    if prediction.lower() == 'no_relation':
                        stats['no_relation'] += 1
                        continue
                    parts = re.split(r'\s+-\s+', prediction, maxsplit=2)
                    if len(parts) != 3:
                        raise ValueError('prediction phải có dạng subject - relation - object')
                    subject, predicate, obj = parts
                else:
                    subject, predicate, obj = row[sub_col], row[rel_col], row[obj_col]
                key = tuple(' '.join(x.lower().replace('_', ' ').split()) for x in (subject, predicate, obj))
                if not all(key):
                    raise ValueError('bộ ba có thành phần rỗng')
                if score < min_conf:
                    stats['below_threshold'] += 1
                    continue
                if key in seen[image_id]:
                    stats['duplicates'] += 1
                    old = next(t for t in table[image_id] if (t['subject'], t['predicate'], t['object']) == key)
                    old['confidence'] = max(old['confidence'], score)
                    continue
                seen[image_id].add(key)
                table[image_id].append(dict(zip(('subject', 'predicate', 'object'), key), confidence=score))
            except (ValueError, TypeError, KeyError, AttributeError) as exc:
                raise ValueError(f'{csv_path}, dòng {row_number}: {exc}') from exc
    stats['images'] = len(table)
    stats['triplets'] = sum(map(len, table.values()))
    return table, stats


def expand_predicate(predicate, max_synonyms=4):
    """Sorted same-synset lemmas of first verb sense; no troponym-as-synonym expansion."""
    base = words(predicate)
    variants = {base}
    variants.update(words(p) for p in _PHRASE_ALIASES.get(' '.join(base), ()))
    if _WN is not None and len(base) == 1:
        synsets = _WN.synsets(base[0], pos=_WN.VERB)
        if synsets:
            variants.update(words(name) for name in synsets[0].lemma_names())
    variants.discard(())
    return (base,) + tuple(sorted(variants - {base})[:max_synonyms])


def phrase_spans(tokens, phrase):
    size = len(phrase)
    return [(i, i + size) for i in range(len(tokens) - size + 1)
            if size and tokens[i:i + size] == phrase]


def compile_triplets(triplets):
    return [dict(t, subject_tokens=words(t['subject']), object_tokens=words(t['object']),
                 predicate_variants=expand_predicate(t['predicate'])) for t in triplets]


def relation_score(text, prepared_triplets, max_gap=12):
    """Ordered S -> P -> O lexical evidence, restricted to one clause. Not full parsing."""
    matched, total = [], 0.0
    clauses = re.split(r'[.!?;]|\b(?:and|but|while|whereas)\b', text.lower())
    clauses = [words(clause) for clause in clauses]
    for triplet in prepared_triplets:
        conf = triplet['confidence']
        total += conf
        hit = False
        for tokens in clauses:
            if any(t in {'not', 'never', 'without', 'no'} or t.endswith("n't") for t in tokens):
                continue
            ss = phrase_spans(tokens, triplet['subject_tokens'])
            oo = phrase_spans(tokens, triplet['object_tokens'])
            pp = [span for variant in triplet['predicate_variants'] for span in phrase_spans(tokens, variant)]
            hit = any(s1 <= p0 and p1 <= o0 and p0-s1 <= max_gap and o0-p1 <= max_gap
                      for s0, s1 in ss for p0, p1 in pp for o0, o1 in oo)
            if hit:
                break
        if hit:
            matched.append(triplet)
    return (sum(t['confidence'] for t in matched) / total if total else 0.0), [
        [t['subject'], t['predicate'], t['object']] for t in matched]


def rerank_candidates(candidates, triplets_by_image, alpha=0.2):
    """Use original normalized L5A baseline_score; alpha=0 preserves scores and stable ranks."""
    if not 0 <= alpha <= 1:
        raise ValueError('alpha phải nằm trong [0,1]')
    groups, seen = defaultdict(list), set()
    prepared = {}
    for raw in candidates:
        item = dict(raw)
        image_id = canonical_image_id(item['image_id'])
        caption_id = str(item['caption_id'])
        if (image_id, caption_id) in seen:
            raise ValueError(f'Caption trùng ID trong cùng ảnh: {image_id}/{caption_id}')
        seen.add((image_id, caption_id))
        if not isinstance(item['text'], str) or not item['text'].strip():
            raise ValueError('Caption rỗng/không phải chuỗi')
        base = float(item['baseline_score'])
        if not math.isfinite(base) or not 0 <= base <= 1:
            raise ValueError('baseline_score phải là điểm L5A đã chuẩn hóa trong [0,1]')
        if item.get('source_image_id') is not None and canonical_image_id(item['source_image_id']) == image_id:
            raise ValueError('Ứng viên có cùng source_image_id với ảnh truy vấn')
        if image_id not in triplets_by_image:
            raise ValueError(f'Thiếu bản ghi RelTR cho ảnh {image_id}; kiểm tra split/image_id')
        if image_id not in prepared:
            prepared[image_id] = compile_triplets(triplets_by_image[image_id]) if alpha else []
        score, matches = relation_score(item['text'], prepared[image_id]) if alpha else (0.0, [])
        final = base if alpha == 0 or not prepared[image_id] else (1-alpha)*base + alpha*score
        item.update(image_id=image_id, baseline_score=base, relation_score=score,
                    final_score=final, matched_triplets=matches)
        groups[image_id].append(item)
    for items in groups.values():
        items.sort(key=lambda item: -item['final_score'])
    return dict(groups)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

## 4. Bật WordNet và kiểm tra hợp đồng xử lý
Các ca nhỏ sau đây sử dụng dữ liệu giả lập. PASS chỉ chứng minh các hành vi được kiểm tra,
không chứng minh điểm CIDEr/SPICE tăng và không thay thế đánh giá COCO.

In [ ]:
configure_wordnet(USE_WORDNET)
print('WordNet:', 'enabled' if USE_WORDNET else 'disabled — chỉ chuẩn hóa và alias tĩnh')
print('Ví dụ biến thể ride:', expand_predicate('riding'))

In [ ]:
def run_self_tests():
    """Synthetic contract tests only; no COCO metrics or training results."""
    import tempfile
    with tempfile.TemporaryDirectory(prefix='lv6_check_') as directory:
        root = Path(directory)
        real = root / 'real.csv'
        real.write_text('image,prediction,score\n'
                        'COCO_val2014_000000000001.jpg,person - chasing - dog,0.9\n'
                        'COCO_val2014_000000000001.jpg,person - chasing - dog,0.8\n'
                        'COCO_val2014_000000000002.jpg,no_relation,0\n'
                        'COCO_val2014_000000000003.jpg,cat - on - table,0.2\n', encoding='utf-8')
        table, stats = load_reltr_triplets(real)
        assert stats['triplets'] == 1 and stats['duplicates'] == 1
        assert table['2'] == [] and table['3'] == []
        alias = root / 'alias.csv'
        alias.write_text('Image_ID,Subject,Relation,Object,Score\n000001,person,chasing,dog,0.9\n', encoding='utf-8')
        assert load_reltr_triplets(alias)[0]['1'] == table['1']
        prepared = compile_triplets(table['1'])
        assert relation_score('A person chasing a dog', prepared)[0] == 1
        assert relation_score('A dog chasing a person', prepared)[0] == 0
        assert relation_score('A person is not chasing a dog', prepared)[0] == 0
        assert relation_score('A person sleeps and a cat chases a dog', prepared)[0] == 0
        spatial = compile_triplets([{'subject':'cat', 'predicate':'on', 'object':'table', 'confidence':0.9}])
        assert relation_score('A cat on a table', spatial)[0] == 1
        assert relation_score('A cat under a table', spatial)[0] == 0
        assert words('ride riding rides ridden') == ('ride',)*4
        candidates = [
            {'image_id':'1','caption_id':'a','text':'A dog chasing a person','baseline_score':0.72},
            {'image_id':'1','caption_id':'b','text':'A person chasing a dog','baseline_score':0.70},
            {'image_id':'2','caption_id':'c','text':'An empty room','baseline_score':0.99},
        ]
        base = rerank_candidates(candidates, table, alpha=0)
        assert [c['caption_id'] for c in base['1']] == ['a','b']
        assert [c['final_score'] for c in base['1']] == [0.72,0.70]
        improved = rerank_candidates(candidates, table, alpha=0.2)
        assert [c['caption_id'] for c in improved['1']] == ['b','a']
        assert len(improved['2']) == 1 and improved['2'][0]['final_score'] == 0.99
        tied = [dict(candidates[0], baseline_score=0.5), dict(candidates[1], baseline_score=0.5)]
        assert [c['caption_id'] for c in rerank_candidates(tied, table, 0)['1']] == ['a','b']
        for invalid in [float('nan'), float('inf'), -1, 2]:
            try:
                rerank_candidates([dict(candidates[0], baseline_score=invalid)], table)
            except ValueError:
                pass
            else:
                raise AssertionError('Phải từ chối điểm không hợp lệ')
        try:
            rerank_candidates([dict(candidates[0], image_id='999')], table)
        except ValueError:
            pass
        else:
            raise AssertionError('Phải phát hiện thiếu image_id')
        try:
            rerank_candidates([dict(candidates[0], source_image_id='1')], table)
        except ValueError:
            pass
        else:
            raise AssertionError('Phải loại ứng viên cùng ảnh')
    print('PASS: CSV thật và alias, ID, ngưỡng confidence, no_relation, chiều S-P-O, phủ định, '
          'ranh giới mệnh đề, quan hệ không gian, baseline alpha=0, tie, nhóm theo ảnh, '
          'điểm lỗi và ứng viên cùng ảnh. Đây là dữ liệu giả lập, không phải benchmark COCO.')


run_self_tests()

## 5. Đọc CSV RelTR thật trên Val
Hỗ trợ cả chuỗi `subject - relation - object` và schema tách cột.
Ảnh `no_relation` hoặc chỉ có bộ ba dưới ngưỡng vẫn được giữ để fallback về baseline.
Đủ 5.000 ảnh là kiểm tra số lượng; để xác nhận đúng Karpathy Val phải đối chiếu thêm danh sách ID chuẩn
trong pipeline Level 5A. Notebook chưa có danh sách đó nên không suy ra đúng split chỉ từ số lượng.

In [ ]:
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Không tìm thấy DATA_ROOT: {DATA_ROOT}. Sửa mục 2 theo Drive của bạn.')
if not RELTR_CSV.is_file():
    raise FileNotFoundError(f'Thiếu CSV: {RELTR_CSV}. Đặt file RelTR Val thật vào đây hoặc sửa mục 2.')
triplets_by_image, reltr_stats = load_reltr_triplets(RELTR_CSV, MIN_CONFIDENCE)
if reltr_stats['images'] != EXPECTED_VAL_IMAGES:
    raise ValueError(f"CSV có {reltr_stats['images']} ảnh; cần {EXPECTED_VAL_IMAGES} ảnh Val. Kiểm tra dữ liệu.")
print(json.dumps(reltr_stats, ensure_ascii=False, indent=2))
preview = list(triplets_by_image.items())[:5]
for image_id, triplets in preview:
    print(image_id, triplets[:3])

## 6. Xuất bản module để tích hợp về sau
File `.py` được tạo tự động trong máy ảo Colab, không ghi đè mã Level 5A trên Drive.
Có thể tải file này để tích hợp vào bước trước MMR sau khi đã kiểm tra contract điểm baseline.

In [ ]:
runtime_dir = Path('/content/lv6_runtime')
runtime_dir.mkdir(parents=True, exist_ok=True)
runtime_module = runtime_dir / 'reltr_relation_reranking.py'
MODULE_SOURCE = '"""Lv6 prototype: ordered lexical relation matching, not a scene-graph parser."""\nimport csv\nimport hashlib\nimport json\nimport math\nimport re\nfrom collections import defaultdict\nfrom pathlib import Path\n\n_WN = None\n_LEMMATIZER = None\n_ALIASES = {\n    \'man\': \'person\', \'woman\': \'person\', \'boy\': \'person\', \'girl\': \'person\',\n    \'child\': \'person\', \'people\': \'person\', \'men\': \'person\', \'women\': \'person\',\n    \'children\': \'person\', \'bike\': \'bicycle\', \'bikes\': \'bicycle\',\n    \'bicycles\': \'bicycle\', \'dogs\': \'dog\', \'cats\': \'cat\', \'horses\': \'horse\',\n    \'riding\': \'ride\', \'rides\': \'ride\', \'ridden\': \'ride\', \'rode\': \'ride\',\n    \'chasing\': \'chase\', \'chases\': \'chase\', \'chased\': \'chase\',\n    \'holding\': \'hold\', \'holds\': \'hold\', \'held\': \'hold\',\n    \'wearing\': \'wear\', \'wears\': \'wear\', \'wore\': \'wear\',\n    \'sitting\': \'sit\', \'sits\': \'sit\', \'sat\': \'sit\',\n    \'standing\': \'stand\', \'stands\': \'stand\', \'stood\': \'stand\',\n    \'playing\': \'play\', \'plays\': \'play\', \'played\': \'play\',\n    \'eating\': \'eat\', \'eats\': \'eat\', \'ate\': \'eat\',\n    \'carrying\': \'carry\', \'carries\': \'carry\', \'carried\': \'carry\',\n}\n_PHRASE_ALIASES = {\n    \'next to\': (\'beside\', \'alongside\'),\n    \'above\': (\'over\',), \'below\': (\'under\', \'beneath\'),\n}\n\n\ndef configure_wordnet(enabled=True):\n    """Fail explicitly when requested WordNet data are unavailable."""\n    global _WN, _LEMMATIZER\n    if not enabled:\n        _WN, _LEMMATIZER = None, None\n        return\n    from nltk.corpus import wordnet\n    from nltk.stem import WordNetLemmatizer\n    wordnet.ensure_loaded()\n    _WN, _LEMMATIZER = wordnet, WordNetLemmatizer()\n\n\ndef canonical_image_id(value):\n    text = str(value).strip().replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1]\n    if re.fullmatch(r\'\\d+\', text):\n        return str(int(text))\n    match = re.fullmatch(r\'COCO_(?:train|val|test)2014_(\\d+)\\.jpg\', text, re.I)\n    if match:\n        return str(int(match.group(1)))\n    raise ValueError(f\'image_id/filename COCO không hợp lệ: {value!r}\')\n\n\ndef words(text):\n    result = []\n    for token in re.findall(r"[a-z]+(?:\'[a-z]+)?", str(text).lower().replace(\'_\', \' \')):\n        token = _ALIASES.get(token, token)\n        if _LEMMATIZER is not None:\n            token = _LEMMATIZER.lemmatize(token, pos=\'v\')\n            token = _LEMMATIZER.lemmatize(token, pos=\'n\')\n        result.append(_ALIASES.get(token, token))\n    return tuple(result)\n\n\ndef load_reltr_triplets(csv_path, min_conf=0.5):\n    """Accept real image/prediction/score CSV and split-column CSV; retain empty images."""\n    if not 0 <= min_conf <= 1:\n        raise ValueError(\'min_conf phải nằm trong [0,1]\')\n    table, seen = {}, defaultdict(set)\n    stats = {\'rows\': 0, \'no_relation\': 0, \'below_threshold\': 0, \'duplicates\': 0}\n    with Path(csv_path).open(encoding=\'utf-8-sig\', newline=\'\') as handle:\n        reader = csv.DictReader(handle)\n        names = {name.strip().lower(): name for name in (reader.fieldnames or [])}\n        def pick(*options):\n            return next((names[name] for name in options if name in names), None)\n        image_col = pick(\'image_id\', \'image\', \'file_name\')\n        pred_col = pick(\'prediction\')\n        sub_col, rel_col, obj_col = pick(\'subject\', \'s\'), pick(\'predicate\', \'relation\', \'rel\', \'p\'), pick(\'object\', \'o\')\n        score_col = pick(\'score\', \'confidence\', \'conf\', \'reltr_score\')\n        if not image_col or not score_col or not (pred_col or (sub_col and rel_col and obj_col)):\n            raise ValueError(f\'CSV thiếu schema RelTR hỗ trợ: {reader.fieldnames}\')\n        for row_number, row in enumerate(reader, 2):\n            try:\n                image_id = canonical_image_id(row[image_col])\n                table.setdefault(image_id, [])\n                stats[\'rows\'] += 1\n                score = float(row[score_col])\n                if not math.isfinite(score) or not 0 <= score <= 1:\n                    raise ValueError(\'confidence phải hữu hạn và thuộc [0,1]\')\n                if pred_col:\n                    prediction = row[pred_col].strip()\n                    if prediction.lower() == \'no_relation\':\n                        stats[\'no_relation\'] += 1\n                        continue\n                    parts = re.split(r\'\\s+-\\s+\', prediction, maxsplit=2)\n                    if len(parts) != 3:\n                        raise ValueError(\'prediction phải có dạng subject - relation - object\')\n                    subject, predicate, obj = parts\n                else:\n                    subject, predicate, obj = row[sub_col], row[rel_col], row[obj_col]\n                key = tuple(\' \'.join(x.lower().replace(\'_\', \' \').split()) for x in (subject, predicate, obj))\n                if not all(key):\n                    raise ValueError(\'bộ ba có thành phần rỗng\')\n                if score < min_conf:\n                    stats[\'below_threshold\'] += 1\n                    continue\n                if key in seen[image_id]:\n                    stats[\'duplicates\'] += 1\n                    old = next(t for t in table[image_id] if (t[\'subject\'], t[\'predicate\'], t[\'object\']) == key)\n                    old[\'confidence\'] = max(old[\'confidence\'], score)\n                    continue\n                seen[image_id].add(key)\n                table[image_id].append(dict(zip((\'subject\', \'predicate\', \'object\'), key), confidence=score))\n            except (ValueError, TypeError, KeyError, AttributeError) as exc:\n                raise ValueError(f\'{csv_path}, dòng {row_number}: {exc}\') from exc\n    stats[\'images\'] = len(table)\n    stats[\'triplets\'] = sum(map(len, table.values()))\n    return table, stats\n\n\ndef expand_predicate(predicate, max_synonyms=4):\n    """Sorted same-synset lemmas of first verb sense; no troponym-as-synonym expansion."""\n    base = words(predicate)\n    variants = {base}\n    variants.update(words(p) for p in _PHRASE_ALIASES.get(\' \'.join(base), ()))\n    if _WN is not None and len(base) == 1:\n        synsets = _WN.synsets(base[0], pos=_WN.VERB)\n        if synsets:\n            variants.update(words(name) for name in synsets[0].lemma_names())\n    variants.discard(())\n    return (base,) + tuple(sorted(variants - {base})[:max_synonyms])\n\n\ndef phrase_spans(tokens, phrase):\n    size = len(phrase)\n    return [(i, i + size) for i in range(len(tokens) - size + 1)\n            if size and tokens[i:i + size] == phrase]\n\n\ndef compile_triplets(triplets):\n    return [dict(t, subject_tokens=words(t[\'subject\']), object_tokens=words(t[\'object\']),\n                 predicate_variants=expand_predicate(t[\'predicate\'])) for t in triplets]\n\n\ndef relation_score(text, prepared_triplets, max_gap=12):\n    """Ordered S -> P -> O lexical evidence, restricted to one clause. Not full parsing."""\n    matched, total = [], 0.0\n    clauses = re.split(r\'[.!?;]|\\b(?:and|but|while|whereas)\\b\', text.lower())\n    clauses = [words(clause) for clause in clauses]\n    for triplet in prepared_triplets:\n        conf = triplet[\'confidence\']\n        total += conf\n        hit = False\n        for tokens in clauses:\n            if any(t in {\'not\', \'never\', \'without\', \'no\'} or t.endswith("n\'t") for t in tokens):\n                continue\n            ss = phrase_spans(tokens, triplet[\'subject_tokens\'])\n            oo = phrase_spans(tokens, triplet[\'object_tokens\'])\n            pp = [span for variant in triplet[\'predicate_variants\'] for span in phrase_spans(tokens, variant)]\n            hit = any(s1 <= p0 and p1 <= o0 and p0-s1 <= max_gap and o0-p1 <= max_gap\n                      for s0, s1 in ss for p0, p1 in pp for o0, o1 in oo)\n            if hit:\n                break\n        if hit:\n            matched.append(triplet)\n    return (sum(t[\'confidence\'] for t in matched) / total if total else 0.0), [\n        [t[\'subject\'], t[\'predicate\'], t[\'object\']] for t in matched]\n\n\ndef rerank_candidates(candidates, triplets_by_image, alpha=0.2):\n    """Use original normalized L5A baseline_score; alpha=0 preserves scores and stable ranks."""\n    if not 0 <= alpha <= 1:\n        raise ValueError(\'alpha phải nằm trong [0,1]\')\n    groups, seen = defaultdict(list), set()\n    prepared = {}\n    for raw in candidates:\n        item = dict(raw)\n        image_id = canonical_image_id(item[\'image_id\'])\n        caption_id = str(item[\'caption_id\'])\n        if (image_id, caption_id) in seen:\n            raise ValueError(f\'Caption trùng ID trong cùng ảnh: {image_id}/{caption_id}\')\n        seen.add((image_id, caption_id))\n        if not isinstance(item[\'text\'], str) or not item[\'text\'].strip():\n            raise ValueError(\'Caption rỗng/không phải chuỗi\')\n        base = float(item[\'baseline_score\'])\n        if not math.isfinite(base) or not 0 <= base <= 1:\n            raise ValueError(\'baseline_score phải là điểm L5A đã chuẩn hóa trong [0,1]\')\n        if item.get(\'source_image_id\') is not None and canonical_image_id(item[\'source_image_id\']) == image_id:\n            raise ValueError(\'Ứng viên có cùng source_image_id với ảnh truy vấn\')\n        if image_id not in triplets_by_image:\n            raise ValueError(f\'Thiếu bản ghi RelTR cho ảnh {image_id}; kiểm tra split/image_id\')\n        if image_id not in prepared:\n            prepared[image_id] = compile_triplets(triplets_by_image[image_id]) if alpha else []\n        score, matches = relation_score(item[\'text\'], prepared[image_id]) if alpha else (0.0, [])\n        final = base if alpha == 0 or not prepared[image_id] else (1-alpha)*base + alpha*score\n        item.update(image_id=image_id, baseline_score=base, relation_score=score,\n                    final_score=final, matched_triplets=matches)\n        groups[image_id].append(item)\n    for items in groups.values():\n        items.sort(key=lambda item: -item[\'final_score\'])\n    return dict(groups)\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n'
runtime_module.write_text(MODULE_SOURCE, encoding='utf-8')
print('Module mới:', runtime_module)
# Muốn tải module: bỏ dấu # ở hai dòng sau.
# from google.colab import files
# files.download(str(runtime_module))

## 7. Định dạng caption ứng viên thật và điểm nối Level 5A
Không phải upload file này nếu chỉ kiểm tra RelTR. Để xếp hạng caption thật, xuất Top-N (ví dụ Top-50)
trước MMR từ pipeline Level 5A thành JSONL, mỗi dòng một ứng viên:

```json
{"image_id": 184613, "caption_id": "kb_row_123", "text": "A boy holding an umbrella", "baseline_score": 0.72, "source_image_id": 98765}
```

Đây chỉ là ví dụ schema, không phải dữ liệu thực nghiệm. `caption_id` phải là khóa ánh xạ đúng về hàng
caption/embedding trong kho Train+Restval; `baseline_score` phải là **điểm relevance L5A trước MMR đã
chuẩn hóa trong [0,1]**, giữ nguyên cách tính CLIP/Jaccard cũ. Không lấy caption tham chiếu Val làm
ứng viên. `source_image_id` nên có để kiểm tra loại ảnh trùng; nếu thiếu, notebook không thể xác minh
nguồn của ứng viên. File phải bao phủ cùng 5.000 ảnh như CSV RelTR.

Điểm mới: `final_score = (1-alpha)*baseline_score + alpha*relation_score`.
Khi `alpha=0`, giữ chính xác điểm gốc và thứ tự ổn định khi đồng điểm. Khi ảnh không có bộ ba đủ tin cậy,
giữ điểm baseline. Không tính lại Jaccard bằng tập vị ngữ ở nhánh baseline.

**Điểm nối cần làm trên code Level 5A:** đưa `final_score` của toàn bộ pool vào MMR hiện hữu, giữ khóa
caption/embedding, rồi chọn đúng 5 caption cho Knowledge Memory. Không lấy Top-5 từ notebook này để
thay MMR. Trước khi kết luận baseline giống hệt, đối chiếu cả kết quả MMR và confidence đầu vào gate;
test ở đây mới kiểm tra hợp đồng điểm và thứ tự trong module này.

In [ ]:
reranked = None
candidate_sha = None
candidate_count = 0
missing_source_count = 0
if CANDIDATES_JSONL is None:
    print('Chưa cấp ứng viên Level 5A: hoàn tất kiểm tra RelTR, chưa chạy reranking caption thật.')
else:
    CANDIDATES_JSONL = Path(CANDIDATES_JSONL)
    candidates = []
    with CANDIDATES_JSONL.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    candidates.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'JSONL lỗi dòng {line_number}') from exc
    if not candidates:
        raise ValueError('File ứng viên rỗng')
    ids = {canonical_image_id(c['image_id']) for c in candidates}
    if ids != set(triplets_by_image):
        raise ValueError(f'ID ứng viên và RelTR lệch: thiếu {len(set(triplets_by_image)-ids)}, thừa {len(ids-set(triplets_by_image))}.')
    candidate_count = len(candidates)
    missing_source_count = sum(c.get('source_image_id') is None for c in candidates)
    if missing_source_count:
        print('Chưa xác minh nguồn/cùng ảnh cho', missing_source_count, 'ứng viên thiếu source_image_id.')
    reranked = rerank_candidates(candidates, triplets_by_image, ALPHA)
    candidate_sha = file_sha256(CANDIDATES_JSONL)
    print('Đã xếp hạng', candidate_count, 'ứng viên của', len(reranked), 'ảnh, riêng từng ảnh.')
    for image_id, items in list(reranked.items())[:2]:
        print('Ảnh', image_id)
        for item in items[:3]:
            print(item['caption_id'], round(item['final_score'], 4), item['text'], item['matched_triplets'])

## 8. Lưu kết quả thử nghiệm vào thư mục mới
Mỗi lần chạy tạo một thư mục riêng trên Drive. Không ghi đè cache hoặc kết quả Level 4/5.
Chỉ lưu điểm reranking, thống kê RelTR, cấu hình và checksum; không ghi điểm COCO giả lập.

In [ ]:
from datetime import datetime, timezone
import uuid
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
run_dir = OUTPUT_ROOT / run_id
run_dir.mkdir(parents=True, exist_ok=False)
manifest = {
    'status':'prototype_val_rerank' if reranked is not None else 'prototype_val_inspect',
    'split':SPLIT, 'alpha':ALPHA, 'min_confidence':MIN_CONFIDENCE,
    'wordnet_enabled':USE_WORDNET, 'nltk_version':nltk.__version__,
    'reltr_csv':str(RELTR_CSV), 'reltr_sha256':file_sha256(RELTR_CSV),
    'candidates_jsonl':str(CANDIDATES_JSONL) if CANDIDATES_JSONL is not None else None,
    'candidates_sha256':candidate_sha, 'candidate_count':candidate_count,
    'missing_source_image_id_count':missing_source_count,
    'module_sha256':file_sha256(runtime_module), 'reltr_stats':reltr_stats,
    'limitations':'Ordered lexical heuristic; no full syntax, decoder, MMR, COCO evaluator or verified Karpathy ID list.'
}
(run_dir/'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
(run_dir/'reltr_preview.json').write_text(json.dumps(dict(preview), ensure_ascii=False, indent=2), encoding='utf-8')
(run_dir/'reltr_relation_reranking.py').write_text(MODULE_SOURCE, encoding='utf-8')
if reranked is not None:
    with (run_dir/'reranked_candidates_val.jsonl').open('w', encoding='utf-8') as handle:
        for items in reranked.values():
            for rank, item in enumerate(items, 1):
                handle.write(json.dumps(dict(item, rank=rank), ensure_ascii=False) + '\n')
print('Đã lưu:', run_dir)
print('Chưa huấn luyện/sinh caption/chấm COCO. Tiếp tục nối vào MMR của Level 5A khi có module thực thi.')

## 9. Điều kiện trước khi báo cáo kết quả
1. Kiểm tra định tính caption đúng quan hệ, đảo vai trò, sai vị ngữ, không gian và câu bị động trên Val.
2. Đối chiếu danh sách Karpathy Val và nguồn caption Train+Restval với pipeline gốc.
3. Với `alpha=0`, đối chiếu toàn bộ cache, lựa chọn MMR và đầu vào decoder với Level 5A.
4. Chọn alpha trên Val. Thử cố định checkpoint là **thay reranking lúc suy luận**; không gọi đó là train mới.
5. Khi triển khai thí nghiệm chính thức, giữ Beam k=3, max_len=20, 5 knowledge tokens và evaluator
   COCO/Java; Test theo cấu hình đã khóa. Mean ± Std dùng ddof=1 và các checkpoint seed hợp lệ.

Notebook này chưa có kết quả chứng minh cải thiện. Tên Level 6 chỉ là mốc tổ chức nhánh nghiên cứu.